# Python 3.15 Data Engineering — Phase 7: Benchmark Results

**Author:** Dr. Ceasar Jackson Jr.  
**Platform:** macOS 26.5 ARM64  
**Python:** 3.15.0b1  
**Environment:** uv · `.venv`  

---

This notebook loads the benchmark CSVs from `data/` and produces publication-quality charts for both benchmark suites.

**Sections**
1. Setup & data loading
2. Pandas vs. Polars — speedup heatmap
3. Pandas vs. Polars — absolute timings
4. DuckDB vs. PyArrow — winner breakdown
5. DuckDB vs. PyArrow — absolute timings
6. Combined findings summary

## 1. Setup & Data Loading

In [ ]:
import sys, os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print(f"Python     : {sys.version.split()[0]}")
print(f"pandas     : {pd.__version__}")
print(f"matplotlib : {matplotlib.__version__}")

# Style
plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.25,
    "grid.linestyle":    "--",
    "font.size":         10,
})

# Color palette — matches project theme
C = {
    "navy":   "#0D1B2A",
    "teal":   "#1D9E75",
    "teal_d": "#0F6E56",
    "blue":   "#185FA5",
    "blue_l": "#378ADD",
    "amber":  "#BA7517",
    "red":    "#A32D2D",
    "gray":   "#888780",
    "gray_l": "#D3D1C7",
}

DATA_DIR = Path("../data")   # notebook lives in notebooks/, data in data/
assert DATA_DIR.exists(), f"data/ directory not found at {DATA_DIR.resolve()}"
print(f"\nData directory: {DATA_DIR.resolve()}")

In [ ]:
# Load benchmark CSVs
pv = pd.read_csv(DATA_DIR / "benchmark_pandas_polars.csv")
da = pd.read_csv(DATA_DIR / "benchmark_duckdb_pyarrow.csv")

print("Pandas vs. Polars CSV:")
display(pv)

print("\nDuckDB vs. PyArrow CSV:")
display(da)

## 2. Pandas vs. Polars — Speedup Heatmap

In [ ]:
# Pivot speedup values: rows = operations, cols = dataset sizes
pivot = pv.pivot(index="operation", columns="size", values="speedup_x")
pivot.columns = [f"{c//1_000:,}K" if c < 1_000_000 else f"{c//1_000_000}M" for c in pivot.columns]
pivot.index = [op.capitalize() for op in pivot.index]

fig, ax = plt.subplots(figsize=(8, 3.5))

im = ax.imshow(pivot.values, cmap="YlGn", aspect="auto", vmin=1, vmax=42)
plt.colorbar(im, ax=ax, label="Polars speedup (×)")

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=11)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=11)
ax.set_title("Polars Speedup over Pandas — Python 3.15.0b1",
             fontsize=12, fontweight="bold", pad=12)
ax.grid(False)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, f"{val:.1f}×", ha="center", va="center",
                fontsize=12, fontweight="bold",
                color="white" if val > 20 else C["navy"])

plt.tight_layout()
plt.savefig(DATA_DIR / "chart_polars_speedup_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to data/chart_polars_speedup_heatmap.png")

## 3. Pandas vs. Polars — Absolute Timings at 5M Rows

In [ ]:
pv_5m = pv[pv["size"] == 5_000_000].copy()
ops   = pv_5m["operation"].tolist()
x     = np.arange(len(ops))
w     = 0.35

fig, ax = plt.subplots(figsize=(10, 4.5))

bars_p = ax.bar(x - w/2, pv_5m["pandas_s"],  w, label="Pandas 3.0.3",  color=C["gray"],   alpha=0.85)
bars_q = ax.bar(x + w/2, pv_5m["polars_s"],  w, label="Polars 1.41.2", color=C["teal"],   alpha=0.85)

# Annotate speedup above the taller bar
for i, row in enumerate(pv_5m.itertuples()):
    taller = max(row.pandas_s, row.polars_s)
    ax.text(i, taller + 0.01, f"{row.speedup_x:.1f}×",
            ha="center", va="bottom", fontsize=9.5,
            fontweight="bold", color=C["teal_d"])

ax.set_xticks(x)
ax.set_xticklabels([o.capitalize() for o in ops], fontsize=11)
ax.set_ylabel("Time (seconds)")
ax.set_title("Pandas vs. Polars — 5M Rows — Python 3.15.0b1",
             fontsize=12, fontweight="bold")
ax.legend(framealpha=0.3)
ax.set_ylim(0, pv_5m[["pandas_s","polars_s"]].max().max() * 1.25)

plt.tight_layout()
plt.savefig(DATA_DIR / "chart_pandas_polars_5m.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to data/chart_pandas_polars_5m.png")

## 4. DuckDB vs. PyArrow — Winner Breakdown

In [ ]:
# Only rows where both timings are available
da_full = da.dropna(subset=["pyarrow_s"]).copy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
sizes = [500_000, 2_000_000, 5_000_000]
size_labels = ["500K rows", "2M rows", "5M rows"]

for ax, size, label in zip(axes, sizes, size_labels):
    sub = da_full[da_full["size"] == size]
    ops_list = sub["operation"].tolist()
    x = np.arange(len(ops_list))

    bar_d = ax.barh(x + 0.2, sub["duckdb_s"],  0.35, color=C["blue"],  alpha=0.85, label="DuckDB")
    bar_a = ax.barh(x - 0.2, sub["pyarrow_s"], 0.35, color=C["teal"],  alpha=0.85, label="PyArrow")

    ax.set_yticks(x)
    ax.set_yticklabels([o.replace("_", " ").capitalize() for o in ops_list], fontsize=9)
    ax.set_xlabel("Seconds")
    ax.set_title(label, fontsize=11, fontweight="bold")
    if ax == axes[0]:
        ax.legend(fontsize=9, framealpha=0.3)
    ax.set_xscale("log")

fig.suptitle("DuckDB (Python 3.15) vs. PyArrow (Docker py314) — Log Scale",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / "chart_duckdb_pyarrow_log.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to data/chart_duckdb_pyarrow_log.png")

## 5. DuckDB vs. PyArrow — Speedup Ratios

In [ ]:
# Positive = DuckDB faster, Negative = PyArrow faster
def signed_speedup(row):
    if row.winner == "DuckDB":
        return row.speedup_x
    else:
        return -row.speedup_x

da_full["signed"] = da_full.apply(signed_speedup, axis=1)

pivot_da = da_full.pivot(index="operation", columns="size", values="signed")
pivot_da.columns = ["500K", "2M", "5M"]
pivot_da.index = [o.replace("_", " ").capitalize() for o in pivot_da.index]

fig, ax = plt.subplots(figsize=(9, 4))
x   = np.arange(len(pivot_da))
w   = 0.25
off = [-w, 0, w]
cols = [C["blue_l"], C["blue"], C["navy"]]

for i, (col_name, color, offset) in enumerate(zip(["500K","2M","5M"], cols, off)):
    vals = pivot_da[col_name].values
    bars = ax.bar(x + offset, vals, w, color=color, alpha=0.85, label=f"{col_name} rows")

ax.axhline(0, color=C["gray"], linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(pivot_da.index, fontsize=10)
ax.set_ylabel("Speedup (+ = DuckDB  − = PyArrow)")
ax.set_title("DuckDB vs. PyArrow — Signed Speedup Ratios",
             fontsize=12, fontweight="bold")
ax.legend(framealpha=0.3)

# Label the extreme values
for i, op in enumerate(pivot_da.index):
    for j, col in enumerate(["500K","2M","5M"]):
        v = pivot_da.loc[op, col]
        if abs(v) >= 20:
            ax.text(i + off[j], v + (3 if v > 0 else -5),
                    f"{abs(v):.0f}×", ha="center", fontsize=8,
                    fontweight="bold",
                    color=C["teal_d"] if v < 0 else C["navy"])

plt.tight_layout()
plt.savefig(DATA_DIR / "chart_duckdb_pyarrow_speedup.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to data/chart_duckdb_pyarrow_speedup.png")

## 6. Combined Findings Summary

In [ ]:
print("=" * 62)
print("Phase 7 Benchmark Summary — Python 3.15.0b1")
print("Author: Dr. Ceasar Jackson Jr.")
print("=" * 62)

print("\n── Pandas vs. Polars ──────────────────────────────────────")
print(f"{'Operation':<12} {'100K':>8} {'1M':>10} {'5M':>10}")
print("-" * 44)
for op in pv["operation"].unique():
    row = pv[pv["operation"] == op].set_index("size")["speedup_x"]
    vals = [f"{row.get(s, float('nan')):.1f}×" for s in [100_000, 1_000_000, 5_000_000]]
    print(f"  {op:<10} {vals[0]:>8} {vals[1]:>10} {vals[2]:>10}")
print("  (all values = Polars speedup over Pandas)")

print("\n── DuckDB vs. PyArrow ─────────────────────────────────────")
print(f"{'Operation':<16} {'500K':>10} {'2M':>12} {'5M':>12}")
print("-" * 54)
for op in da_full["operation"].unique():
    sub = da_full[da_full["operation"] == op].set_index("size")
    parts = []
    for s in [500_000, 2_000_000, 5_000_000]:
        if s in sub.index:
            r = sub.loc[s]
            winner = "D" if r.winner == "DuckDB" else "P"
            parts.append(f"{r.speedup_x:.1f}×({winner})")
        else:
            parts.append("—")
    print(f"  {op:<14} {parts[0]:>10} {parts[1]:>12} {parts[2]:>12}")
print("  D = DuckDB faster  P = PyArrow faster")

print("\n── Verdict ────────────────────────────────────────────────")
print("  Polars  : faster than Pandas on ALL operations at ALL sizes")
print("  DuckDB  : wins Parquet I/O (up to 12×) and aggregation (up to 18×)")
print("  PyArrow : wins filter (up to 37×) and join (up to 77×)")
print("  Recommendation: Polars + DuckDB as the Python 3.15 default stack")
print("=" * 62)